# 安装依赖库

In [ ]:
!pip install mindnlp==0.4.0
!pip uninstall mindformers -y
!pip install transformers==4.40.0
!pip install mindspore==2.4.1

# 导入依赖库

In [ ]:
from mindnlp.transformers import AutoModelForCausalLM, AutoTokenizer
import mindspore as ms
import pandas as pd
import re
import gc

ms.set_context(device_target="Ascend")

# 导入模型与分词器

In [ ]:
model_id = "Qwen/Qwen2.5-3B"
tokenizer = AutoTokenizer.from_pretrained(model_id, ms_dtype=ms.float16, mirror='modelscope')
model = AutoModelForCausalLM.from_pretrained(model_id, ms_dtype=ms.float16, mirror='modelscope')
tokenizer.pad_token_id = tokenizer.eos_token_id

# 辅助函数

### extract_first_uppercase：提取最终结果（最后一个大写字母）

In [ ]:
# 修改提取函数为提取 assistant 部分的最后一个大写字母
def extract_first_uppercase_from_assistant(text):
    uppercase_letters = re.findall(r'[A-Z]', text)
    return uppercase_letters[-1] if uppercase_letters else None

### extract_last_assistant_reply：提取第一次大模型生成的回答

In [ ]:
def extract_last_assistant_reply(text):
    if 'assistant\n' in text:
        return text.split('assistant\n')[-1].strip()
    if 'assistant' in text:
        return text.split('assistant')[-1].strip()
    return text.strip()

# 推理函数

In [ ]:
def predict_cqa_batch(questions, gold_labels, batch_size=16,output_file="cqa_training_results.txt"):
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("")

    tokenizer.padding_side = 'left'
    all_predictions = []
    all_outputs = []
    num_questions = len(questions)

    for start_idx in range(0, num_questions, batch_size):
        gc.collect()
        batch_q = questions[start_idx: start_idx + batch_size]
        batch_gold = gold_labels[start_idx: start_idx + batch_size]

        messages_step1 = [
            [
                {'role': 'system', 'content': "You are a helpful assistant."},  
                {'role': 'user', 'content': f"{q} Let's think step by step."}   
            ]
            for q in batch_q
        ]

        input_ids_1 = tokenizer.apply_chat_template(
            messages_step1, add_generation_prompt=True, return_tensors="ms", tokenize=True,
            padding=True, truncation=True, max_length=1024
        ) 
        outputs_1 = model.generate(input_ids_1, max_new_tokens=512, temperature=0.7, top_p=0.95) 
        step1_texts = tokenizer.batch_decode(outputs_1, skip_special_tokens=True)
        step1_replies = [extract_last_assistant_reply(txt) for txt in step1_texts]

        # 与gsm8k不同的是：cqa的第二个提示词有不同之处
        messages_step2 = [
            [
                {'role': 'system', 'content': "You are a helpful assistant."},
                {'role': 'user', 'content': f"{q} Let's think step by step. {reply} Therefore, among A through E, the answer is"}
            ]
            for q, reply in zip(batch_q, step1_replies)
        ]

        input_ids_2 = tokenizer.apply_chat_template(
            messages_step2, add_generation_prompt=True, return_tensors="ms", tokenize=True,
            padding=True, truncation=True, max_length=1024
        )#把消息转化为模型输入
        outputs_2 = model.generate(input_ids_2, max_new_tokens=256, temperature=0.7, top_p=0.95)
        answer_texts = tokenizer.batch_decode(outputs_2, skip_special_tokens=True)
        predictions = [extract_first_uppercase_from_assistant(ans) for ans in answer_texts]

        # 立即打印当前 batch 的结果
        for q, gold, output, pred in zip(batch_q, batch_gold, answer_texts, predictions):
            print(f"\nQ: {q}\nGold: {gold}\nPred: {pred}\nOutput: {output}\n")

        # 将当前 batch 的结果追加写入文件
        with open(output_file, "a", encoding="utf-8") as f:
            for q, gold, output, pred in zip(batch_q, batch_gold, answer_texts, predictions):
                f.write(f"Q: {q}\nGold: {gold}\nPred: {pred}\nOutput: {output}\n\n")

        all_outputs.extend(answer_texts)
        all_predictions.extend(predictions)

    return all_outputs, all_predictions



# 评估函数

In [ ]:
def evaluate_cqa(max_samples=50, batch_size=16, data_path='data/cqa.parquet'):
    df = pd.read_parquet(data_path)[:max_samples] # 读入前50条数据
    gold_labels = df['answerKey'].tolist()  # 获取答案列表
    questions = df['question'].tolist()  # 获取问题列表

    choices = df['choices'].tolist()  # 获取 choices 列
    choices_list = []
    for choices_iter in choices:
        labels = choices_iter['label']
        texts = choices_iter['text']
        joined = "\n".join([f"({label}) {text}" for label, text in zip(labels, texts)])
        choices_list.append(joined)

    input = []
    for question, choice in zip(questions, choices_list):
        input.append(f"Q: {question}\n {choice}\nA:")  # 构建输入格式
    questions = input
    outputs, predictions = predict_cqa_batch(questions, gold_labels, batch_size=batch_size ,output_file="cqa_training_results.txt")
    accuracy = sum(pred == gold for pred, gold in zip(predictions, gold_labels)) / len(gold_labels)
    print("CommonsenseQA准确率:", accuracy)
    return outputs, gold_labels, accuracy

# 运行函数

In [ ]:
gsm8k_outputs, gsm8k_labels, gsm8k_acc = evaluate_cqa(max_samples=50, batch_size=4, data_path='data/cqa.parquet')